# CP2 Week 13 -- Integration on Larger Data

**Course:** Computer Programming 2 (CP2)
**Prerequisites:** Weeks 1-12
**Focus:** larger datasets, robust error handling, polished exports

## Learning Objectives
- Run your complete pipeline on larger data
- Handle errors gracefully at every stage
- Verify all outputs are generated correctly
- Fix any remaining issues before v2 release

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Full Integration Test

Time to test your entire pipeline end-to-end with realistic data.

In [ ]:
import os, json, random

def run_integration_test(config):
    """Run the complete pipeline and verify all outputs."""
    print("=== Integration Test ===\n")
    errors = []
    
    # Setup directories
    for d in ["data/raw", "data/cleaned", "reports/figures"]:
        path = os.path.join(config.get("base_dir", "."), d)
        os.makedirs(path, exist_ok=True)
    
    # Create larger test dataset
    random.seed(42)
    test_data = []
    for i in range(200):
        val = random.gauss(50, 20)
        test_data.append({"id": i, "value": str(round(val, 2))})
    # Add some bad data
    test_data.append({"id": 200, "value": ""})
    test_data.append({"id": 201, "value": "abc"})
    test_data.append({"id": 202, "value": "999"})
    
    print("Test dataset: " + str(len(test_data)) + " rows")
    
    # Stage 1: Clean
    cleaned = []
    for row in test_data:
        try:
            v = float(row["value"])
            if 0 <= v <= 100:
                cleaned.append({**row, "value": v})
        except (ValueError, TypeError):
            pass
    print("Cleaned: " + str(len(cleaned)) + " rows")
    
    # Stage 2: Analyze
    values = [r["value"] for r in cleaned]
    mean_v = sum(values) / len(values)
    std_v = (sum((x - mean_v)**2 for x in values) / len(values))**0.5
    results = {
        "analysis_summary": {
            "count": len(values),
            "mean": round(mean_v, 2),
            "std": round(std_v, 2),
            "min": round(min(values), 2),
            "max": round(max(values), 2),
        }
    }
    
    # Verify
    assert len(cleaned) > 100, "Too many rows dropped"
    assert len(results["analysis_summary"]) >= 3, "Need 3+ metrics"
    
    print("\nResults: " + str(results["analysis_summary"]))
    print("\n=== Integration Test PASSED ===")

config = {"base_dir": "."}
run_integration_test(config)

**Expected Output:**
```
=== Integration Test ===

Test dataset: 203 rows
Cleaned: ~160 rows

Results: {'count': ~160, 'mean': ~50, ...}

=== Integration Test PASSED ===
```

---
## Part 2: Error Recovery

A robust pipeline does not crash on bad data -- it recovers and reports.

In [ ]:
def robust_pipeline(data, config):
    """Run pipeline with error recovery at every stage."""
    print("=== Robust Pipeline ===")
    errors = []
    results = {}
    
    # Stage 1: Validate
    try:
        if not data:
            raise ValueError("Empty data")
        actual_cols = set(data[0].keys())
        req_cols = set(config.get("required_columns", []))
        missing = req_cols - actual_cols
        if missing:
            raise ValueError("Missing columns: " + str(sorted(missing)))
        print("[OK] Validation passed")
    except Exception as e:
        errors.append("validate: " + str(e))
        print("[ERR] Validation: " + str(e))
    
    # Stage 2: Clean
    try:
        cleaned = []
        for row in data:
            try:
                v = float(row.get("value", ""))
                if 0 <= v <= 100:
                    cleaned.append({**row, "value": v})
            except (ValueError, TypeError):
                pass
        results["n_clean"] = len(cleaned)
        print("[OK] Cleaning: " + str(len(data)) + " -> " + str(len(cleaned)))
    except Exception as e:
        errors.append("clean: " + str(e))
        print("[ERR] Cleaning: " + str(e))
        cleaned = []
    
    # Stage 3: Analyze
    try:
        if not cleaned:
            raise ValueError("No clean data to analyze")
        values = [r["value"] for r in cleaned]
        mean_v = sum(values) / len(values)
        results["analysis_summary"] = {
            "count": len(values),
            "mean": round(mean_v, 2),
            "min": round(min(values), 2),
            "max": round(max(values), 2),
        }
        print("[OK] Analysis: mean=" + str(results["analysis_summary"]["mean"]))
    except Exception as e:
        errors.append("analyze: " + str(e))
        print("[ERR] Analysis: " + str(e))
    
    # Summary
    print("\nPipeline complete: " + str(len(errors)) + " error(s)")
    if errors:
        for err in errors:
            print("  ! " + err)
    
    return results, errors

# Test with good data
import random
random.seed(42)
data = [{"id": i, "value": str(round(random.gauss(50, 20), 2))}
        for i in range(100)]
config = {"required_columns": ["id", "value"]}
results, errors = robust_pipeline(data, config)

**Expected Output:**
```
=== Robust Pipeline ===
[OK] Validation passed
[OK] Cleaning: 100 -> ~75
[OK] Analysis: mean=~50

Pipeline complete: 0 error(s)
```

### Testing with Bad Data

In [ ]:
# Test with completely empty data
print("--- Test: Empty data ---")
results, errors = robust_pipeline([], {"required_columns": ["id"]})
print("Errors:", errors)
print()

# Test with data missing required columns
print("--- Test: Wrong columns ---")
bad_data = [{"x": 1, "y": 2}]
results, errors = robust_pipeline(bad_data, {"required_columns": ["id", "value"]})

---
## Part 3: Output Verification Checklist

In [ ]:
def verify_all_outputs(base_dir="."):
    """Check that all expected outputs exist."""
    print("=== Output Verification ===\n")
    
    checks = [
        ("data/cleaned/cleaned.csv", "Cleaned data"),
        ("reports/report.json", "JSON report"),
        ("reports/report.md", "Markdown report"),
        ("reports/figures/timeseries.png", "Time series plot"),
        ("reports/figures/summary.png", "Summary plot"),
    ]
    
    passed = 0
    failed = 0
    
    for filepath, description in checks:
        full_path = os.path.join(base_dir, filepath)
        if os.path.exists(full_path) and os.path.getsize(full_path) > 0:
            size = os.path.getsize(full_path)
            print("  [OK] " + description + " (" + str(size) + " bytes)")
            passed += 1
        else:
            print("  [MISSING] " + description + " - " + filepath)
            failed += 1
    
    print("\n" + str(passed) + "/" + str(passed + failed) + " outputs verified")
    if failed > 0:
        print("Run your full pipeline to generate missing outputs.")
    return failed == 0

verify_all_outputs()

### Key Takeaway

- Integration testing runs ALL stages together
- Use larger datasets (200+ rows) to expose edge cases
- Verify all output files are created and non-empty
- Robust pipelines recover from errors, they do not crash
- Fix all issues BEFORE the v2 release next week

---
## Homework: 12 Exercises

### Review (1-4)

In [ ]:
# HW1: Run your full pipeline end-to-end.


In [ ]:
# HW2: Verify all output files exist and are non-empty.


In [ ]:
# HW3: Run golden test AND integration test together.


In [ ]:
# HW4: List any remaining issues to fix before v2 release.


### Practice (5-8)

In [ ]:
# HW5: Test with 500+ rows of data.


In [ ]:
# HW6: Add error handling to every pipeline stage.


In [ ]:
# HW7: Verify report.json has all required fields.


In [ ]:
# HW8: Verify all figures are saved correctly.


### Challenge (9-11)

In [ ]:
# HW9: Time the full pipeline and add timing to report.


In [ ]:
# HW10: Test with intentionally bad data (all missing, all outliers).


In [ ]:
# HW11: Create a "pipeline status" dashboard.


### Mini-Project

In [ ]:
# HW12: Run full pipeline, generate all outputs, verify everything.
# Prepare for v2 demo next week.


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)